In [1]:
import numpy as np
import glob
import xarray as xr
import pandas as pd
from shapely.geometry import Point, Polygon
#from storm_functions import latlon2km

# a function to check whether a given point is in the polygon or not (from storm_stats.py)
def in_or_out( lon_in, lat_in, poly_in):
        pt = Point(lon_in,lat_in)
        point_in = pt.within(poly_in)
        return point_in

In [3]:
#################################################################
# "True" means that we will create composites based on track length
track_length = False
storm_width  = 10
#precip_intensity=False
#################################################################
    
for year in np.arange(1960,2022):
    print(year)
    # Load storm data
    sdir  ='/nfs/turbo/seas-hutsona/shansen/etc_tracks/'
    fname=f'storm_track_slp_{year}.npz'
    data = np.load(sdir+fname,allow_pickle=True,encoding='latin1')
    storms = data['storms']

    # Load ERA Data
    edir = '/nfs/turbo/seas-hutsona/shansen/era5_ETC_data/'
    ename= f'era5_{year-1}-{year}.nc'
    edata= xr.open_dataset(edir+ename)
    datetime = pd.to_datetime(edata.time)

    # define polygon for Great Lakes storms (from storm_stats.py)
    lats_polygon = [ 50, 50, 41, 41 ]
    lons_polygon = [ -75.5, -93.5, -93.5, -75.5 ]
    xy = zip(lons_polygon,lats_polygon)
    poly = Polygon(xy)

    count = 1 # required to initialize concatenated arrays

    print('initiated composite for' + str(year))
        
    # loop over tracked storms
    for ed in range(len(storms)):
        flagog = False # still need this "OG flag" to get through criteria below
        flags= [] # keeps track of which storm points are within great lakes region

        # loop over data points on each storm track
        for nl in range(len(storms[ed]['lon'])):
            # flag will remain false if storm point is not in GL at this time
            flag = False
            
            # judge if a point is in the polygon by calling the in_or_out function
            if ( in_or_out(storms[ed]['lon'][nl],storms[ed]['lat'][nl], poly) ):
                flag  = True
                flagog= True
            
            flags.append(flag)

        # do the following code only for storm tracks that pass through GLR
        if (flagog):
 
            # find the date/time at which storm *in* the GLR is at minimum pressure
            min_idx = np.argmin(storms[ed]['amp'][flags])
           
            # save pressure, temperature, wind, and precip at date/time of storm with minimum pressure
            yr,mth = storms[ed]['year'][flags][min_idx],storms[ed]['month'][flags][min_idx]
            dy,hr  = storms[ed]['day'][flags][min_idx],storms[ed]['hour'][flags][min_idx]
            datebool     = (datetime.year==yr)&(datetime.month==mth)&(datetime.day==dy)&(datetime.hour==hr)
          
            print(yr,mth,dy,hr)
            dtes = []
            dtes_init = datetime[datebool]  
            dtes.extend(dtes_init)
            msl = edata.msl.data[datebool]
            t2m = edata.t2m.data[datebool]
            tp  = edata.tp.data[datebool]
            d2m = edata.d2m.data[datebool]
            tcwv= edata.tcwv.data[datebool] # total column vertical integrated water content (kg m^-2)
 

            # find the index of ERA5 lat array and lon array containing center of desired storm point
            elat,elon = edata.latitude.data,edata.longitude.data
            sloc_lat  = np.argwhere(elat==storms[ed]['lat'][flags][min_idx])[0][0]
            sloc_lon  = np.argwhere(elon==storms[ed]['lon'][flags][min_idx])[0][0]

            # grab variables in 550km^2 square centered on desired storm location
            #lati = 20  #20 grid points ~= 550 km in latitude direction 
            #loni = 26  #26 grid points ~= 550 km in longitude direction
            lati = int(storm_width/0.25)
            loni = int(storm_width/0.25)

            msl_c = msl[0,sloc_lat-lati:sloc_lat+(lati+1),sloc_lon-loni:sloc_lon+(loni+1)]
            t2m_c = t2m[0,sloc_lat-lati:sloc_lat+(lati+1),sloc_lon-loni:sloc_lon+(loni+1)]
            tp_c  = tp[0,sloc_lat-lati:sloc_lat+(lati+1),sloc_lon-loni:sloc_lon+(loni+1)]
            d2m_c = d2m[0,sloc_lat-lati:sloc_lat+(lati+1),sloc_lon-loni:sloc_lon+(loni+1)]
            tcwv_c= tcwv[0,sloc_lat-lati:sloc_lat+(lati+1),sloc_lon-loni:sloc_lon+(loni+1)]
           
            west_east  = np.arange(-loni/4,(loni+1)/4,0.25)*85
            south_north = np.arange(-lati/4,(lati+1)/4,0.25)*111
            
            msl_c = np.expand_dims(msl_c,0)
            t2m_c = np.expand_dims(t2m_c,0)
            tp_c  = np.expand_dims(tp_c,0)
            d2m_c = np.expand_dims(d2m_c,0)
            tcwv_c = np.expand_dims(tcwv_c,0)
            
            # NOTE:THIS MIGHT BE WRONG!!!!!!!!!!!!!!!!!!!!!!!
            t2m_rev = t2m_c[::-1,:]
            msl_rev = msl_c[::-1,:]
            tp_rev  = tp_c[::-1,:]
            d2m_rev = d2m_c[::-1,:]
            tcwv_rev= tcwv_c[::-1,:]
            
            dim_dict = {'storm': np.arange(1)}
    
            dtes_array = np.array(dtes)
            dtes_data = xr.DataArray(dtes_array, dims=('storm',), coords={'storm': np.arange(1)})
        
            avg_t2m = np.nanmean(t2m_rev, axis=(1, 2)) #{'units':'K'}
            t2m_data = xr.DataArray(avg_t2m, dims=('storm',), coords={'storm': np.arange(1)})
    
#             avg_msl= np.nanmin(msl_rev,axis=(1,2))-np.nanmean(msl_rev,axis=(1,2))    #{'units':'Pa'}
            avg_msl= np.nanmean(msl_rev,axis=(1,2))    #{'units':'Pa'}
            msl_data = xr.DataArray(avg_msl, dims=('storm',), coords={'storm': np.arange(1)})
    
            avg_tp= np.nanmean(tp_rev,axis=(1,2))      #{'units':'m'})
            tp_data = xr.DataArray(avg_tp, dims=('storm',), coords={'storm': np.arange(1)})
    
            avg_d2m= np.nanmean(d2m_rev,axis=(1,2))    #{'units':'K'}
            d2m_data = xr.DataArray(avg_d2m, dims=('storm',), coords={'storm': np.arange(1)})
    
            avg_tcwv= np.nanmean(tcwv_rev,axis=(1,2))  #{'units':'kg m**-2'}
            tcwv_data = xr.DataArray(avg_tcwv, dims=('storm',), coords={'storm': np.arange(1)})
    
                     
            data_dict = {'Date': dtes_data, 't2m': t2m_data, 'msl': msl_data, 'tp': tp_data, 
                 'd2m': d2m_data, 'tcwv': tcwv_data} 
            #data_dict2 = dict(Date = (dtes_data), t2m= (t2m_data,{'units':'K'}), msl= (msl_data,{'units':'Pa'}),
                              #tp= (tp_data, {'units':'m'}), d2m= (d2m_data, {'units':'K'}),
                              #tcwv= (tcwv_data, {'units':'kg m**-2'})
            
            ds = xr.Dataset(data_vars=data_dict, coords = dim_dict)
            outname = f'storm_{storm_width}deg_{year-1}-{year}_{str(count)}.nc'
            turbodir = '/nfs/turbo/seas-hutsona/shansen/sh_outputs/1d_storms/'
            ds.to_netcdf(turbodir + outname)
            
            # create new dimension to concatenate each storm's data onto one array
            #msl_c = np.expand_dims(msl_c,0)
            #t2m_c = np.expand_dims(t2m_c,0)
            #tp_c  = np.expand_dims(tp_c,0)
            #d2m_c = np.expand_dims(d2m_c,0)
            #tcwv_c = np.expand_dims(tcwv_c,0)
 
            # Initialize arrays for first storm
            #if count==0:             
                    #msl_all = msl_c
                    #t2m_all = t2m_c
                    #tp_all  = tp_c
                    #d2m_all = d2m_c
                    #tcwv_all= tcwv_c
                
                    #newflag=1 #indicates new arrays have initialized
            #else:
                #msl_all = np.concatenate((msl_all,msl_c),axis=0)
                #t2m_all = np.concatenate((t2m_all,t2m_c),axis=0)
                #tp_all  = np.concatenate((tp_all,tp_c),axis=0)
                #d2m_all = np.concatenate((d2m_all,d2m_c),axis=0)
                #tcwv_all= np.concatenate((tcwv_all,tcwv_c),axis=0)
            # NOT concatenating new storm on previous storms' arrays
            
            # dtes.extend(dtes_init) only use for composites...

            count = count + 1
            
        
    #if precip_intensity is False:
        # Save averaged storm composite data for whole season

        # coordinates centered on middle of storm (in meters; negative is west/south, positive is north/east)
    
    # storm_number = np.shape(t2m_all)[0]

        # SYDNIE EDIT THIS BLOCK OF CODE HERE!!!!
        

    
    
   
    #### --------------------------------------------------------------------------------------------------!!!
    #### Here I just split up the original data_dict and kept having trouble with conflicting 
    #### dimensions and coordinates. 
 
    #data_dict = dict(date = (dtes_data), t2m = (t2m_data, {'units':'K'}), msl = (msl_data, {'units':'Pa'}),
                     #tp = (tp_data, {'units':'m'}), d2m = (d2m_data, {'units':'K'}),
                     #tcwv = (tcwv_data, {'units':'kg m**-2'}))

    
     ###-------------------------------------------------------!!!
    #data_dict = dict(date=(["count"],dtes_all), 
                        # t2m=(["count"], np.nanmean(t2m_rev,axis=(1,2)),{'units':'K'}),
                        # msl=(["count"], np.nanmean(msl_rev,axis=(1,2)),{'units':'Pa'}),
                        # tp =(["count"], np.nanmean(tp_rev,axis=(1,2)),{'units':'m'}),
                        # d2m=(["count"], np.nanmean(d2m_rev,axis=(1,2)),{'units':'K'}),
                        # tcwv=(["count"], np.nanmean(tcwv_rev,axis=(1,2)),{'units':'kg m**-2'}))
    #dim_dict  = dict(count = count)


1960
initiated composite for1960
1959 10 9 6
1959 10 11 12
1959 10 16 12
1959 10 24 18
1959 10 26 18
1959 11 11 0
1959 11 24 6
1959 11 25 6
1960 1 3 12
1960 1 15 18
1960 2 7 0
1960 2 8 12
1960 2 11 0
1960 2 22 0
1960 2 26 6
1960 3 17 0
1960 3 17 18
1960 3 22 6
1960 3 30 18
1961
initiated composite for1961
1960 11 16 12
1960 11 29 6
1960 12 6 0
1960 12 16 12
1960 12 21 12
1961 1 8 6
1961 2 19 12
1961 2 26 18
1961 3 5 0
1961 3 9 6
1961 3 15 0
1961 3 21 18
1961 3 23 12
1961 3 27 6
1961 3 27 18
1962
initiated composite for1962
1961 10 14 0
1961 11 2 18
1961 12 4 12
1961 12 8 18
1961 12 10 18
1961 12 13 0
1961 12 20 0
1962 1 7 12
1962 2 5 12
1962 2 14 12
1962 2 16 6
1962 2 19 18
1962 2 17 0
1962 2 22 12
1962 2 26 18
1962 3 12 18
1962 3 22 18
1962 3 28 18
1963
initiated composite for1963
1962 10 12 12
1962 10 23 6
1962 12 7 6
1962 12 9 18
1963 1 9 0
1963 2 20 18
1963 3 17 6
1963 3 20 6
1963 3 27 6
1964
initiated composite for1964
1963 11 29 12
1963 12 8 12
1963 12 9 12
1963 12 13 12
1964 1 9

initiated composite for1999
1998 10 12 12
1998 11 15 6
1998 11 19 18
1998 11 24 0
1998 12 30 0
1999 1 4 0
1999 1 18 18
1999 1 19 6
1999 1 24 6
1999 2 12 6
1999 2 17 12
1999 2 28 0
1999 3 2 18
1999 3 17 18
1999 3 21 18
2000
initiated composite for2000
1999 10 23 0
1999 11 3 6
1999 11 9 18
1999 11 24 12
1999 11 27 12
1999 12 16 6
2000 1 11 12
2000 1 16 6
2000 2 3 6
2000 3 9 6
2000 3 27 12
2000 3 29 6
2001
initiated composite for2001
2000 11 7 0
2000 11 10 0
2000 11 10 0
2000 11 14 12
2000 11 16 12
2000 12 17 18
2001 1 5 12
2001 1 30 18
2001 2 10 0
2001 2 25 12
2001 3 3 0
2001 3 13 18
2001 3 24 0
2002
initiated composite for2002
2001 10 6 6
2001 10 14 6
2001 10 25 12
2001 10 24 0
2001 11 25 0
2001 11 27 12
2001 12 6 0
2002 1 14 6
2002 2 10 12
2002 2 16 0
2002 2 21 0
2002 2 24 0
2002 2 27 6
2002 3 3 12
2002 3 10 0
2002 3 15 0
2002 3 31 12
2002 3 30 6
2003
initiated composite for2003
2002 10 7 6
2002 10 29 18
2002 11 10 18
2002 11 24 0
2002 11 30 0
2002 12 19 6
2002 12 20 12
2003 1 20 6
200

In [4]:
msl_data

<xarray.DataArray (storm: 1)>
array([-1606.1016], dtype=float32)
Coordinates:
  * storm    (storm) int64 0